# Notebook 4: DINOv3 — Other Vision Tasks Overview

This notebook provides a **high-level overview** of the other vision tasks
that DINOv3 excels at, beyond segmentation. We don't go deep into
implementation — instead, we summarize the paper's results and, where
possible, show quick demonstrations using the pretrained model.

## Tasks Covered
1. **Image Classification** (Section 6.2.1) — Linear probing on ImageNet
2. **Instance Retrieval** (Section 6.2.2) — Cosine similarity search
3. **Monocular Depth Estimation** (Section 6.3.3) — Depth Anything V2
4. **Object Detection** (Section 6.3.1) — Plain-DETR with frozen backbone
5. **Video Segmentation Tracking** (Section 6.1.5) — Non-parametric label propagation
6. **3D Correspondence Estimation** (Section 6.1.3) — Keypoint matching
7. **Unsupervised Object Discovery** (Section 6.1.4) — TokenCut

## Setup

In [ ]:
import torch, os, sys, numpy as np, matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(0, 'dinov3')
sys.path.insert(0, 'dinov3-segmentation-study')

## 1. Image Classification (Section 6.2.1)

DINOv3 achieves **88.4% top-1 accuracy** on ImageNet-1k with a linear probe on the CLS token.
This matches weakly-supervised models like SigLIP 2 and PE — a first for SSL.

**Key Results (Table 7)**:

| Model | Type | IN-1k | ObjectNet | IN-Rendition |
|-------|------|-------|-----------|-------------|
| DINOv3 ViT-7B | SSL | 88.4 | 79.0 | 91.1 |
| PEcore ViT-G | WSL | 89.3 | 80.2 | 92.2 |
| SigLIP 2 ViT-g | WSL | 89.1 | 78.6 | 92.2 |
| DINOv2 ViT-g | SSL | 87.3 | 66.4 | 81.1 |

DINOv3 closes the classification gap with WSL models while massively outperforming
them on dense tasks. On ImageNet-C (corruption robustness), DINOv3 is actually the **best**.

## 2. Instance Retrieval (Section 6.2.2)

DINOv3 excels at instance-level image retrieval using cosine similarity of CLS tokens.

**Key Results (Table 9)**:

| Model | Oxford-H mAP | Paris-H mAP | Met GAP | AmsterTime mAP |
|-------|-------------|-------------|---------|---------------|
| DINOv3 | **60.7** | **87.1** | **55.4** | **56.5** |
| DINOv2 | 58.2 | 84.6 | 44.6 | 48.9 |
| AM-RADIO | 50.7 | 85.3 | 30.5 | 46.7 |
| SigLIP 2 | 25.1 | 60.9 | 0.0 | 15.5 |

SSL models dominate retrieval — weakly-supervised models like SigLIP 2 fail badly here.
This makes DINOv3 ideal for applications like image search, deduplication, and visual databases.

In [ ]:
# Quick demo: compute CLS token similarity between images
from src.model_loading import load_backbone
from torchvision.transforms import v2

backbone = load_backbone('vitl16', weights_path='weights/dinov3_vitl16.pth')

def get_cls_embedding(img_path_or_url):
    """Extract the CLS token embedding from an image."""
    if img_path_or_url.startswith('http'):
        import requests
        from io import BytesIO
        img = Image.open(BytesIO(requests.get(img_path_or_url).content)).convert('RGB')
    else:
        img = Image.open(img_path_or_url).convert('RGB')
    
    transform = v2.Compose([
        v2.ToImage(), v2.Resize((512, 512), antialias=True),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])
    tensor = transform(img).unsqueeze(0).cuda()
    
    with torch.no_grad():
        output = backbone.get_intermediate_layers(
            tensor, n=[23], reshape=False, return_class_token=True
        )
        cls_token = output[0][1]  # (1, 1024)
    
    cls_norm = cls_token / cls_token.norm(dim=-1, keepdim=True)
    return cls_norm.squeeze(), img


# Compare 4 images: two cats (similar), one kitchen, one giraffe (different)
urls = [
    'http://images.cocodataset.org/val2017/000000039769.jpg',  # Cats
    'http://images.cocodataset.org/val2017/000000222564.jpg',  # Another cat scene
    'http://images.cocodataset.org/val2017/000000397133.jpg',  # Kitchen
    'http://images.cocodataset.org/val2017/000000087038.jpg',  # Giraffe
]
labels = ['Cats on couch', 'Cat scene', 'Kitchen', 'Giraffe']

embeddings = []
images = []
for url in urls:
    emb, img = get_cls_embedding(url)
    embeddings.append(emb)
    images.append(img)

# Compute pairwise cosine similarity
n = len(embeddings)
sim_matrix = torch.zeros(n, n)
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = (embeddings[i] @ embeddings[j]).item()

# Display
fig, axes = plt.subplots(1, n + 1, figsize=(4 * (n + 1), 4))
for i in range(n):
    axes[i].imshow(images[i].resize((200, 200)))
    axes[i].set_title(labels[i], fontsize=10)
    axes[i].axis('off')

im = axes[n].imshow(sim_matrix.numpy(), cmap='RdYlGn', vmin=0, vmax=1)
axes[n].set_xticks(range(n))
axes[n].set_yticks(range(n))
axes[n].set_xticklabels([l[:6] for l in labels], fontsize=8, rotation=45)
axes[n].set_yticklabels([l[:6] for l in labels], fontsize=8)
axes[n].set_title('CLS Cosine Similarity')

for i in range(n):
    for j in range(n):
        axes[n].text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print('\nSimilar images (both cats) have high cosine similarity.')
print('Different scenes have lower similarity — showing the CLS token captures semantics.')

## 3. Monocular Depth Estimation (Section 6.3.3)

DINOv3 combined with Depth Anything V2 achieves **state-of-the-art** relative depth estimation
on all 5 benchmarks — with a **frozen backbone** (all baselines finetune theirs).

**Key Results (Table 12)**:

| Method | NYUv2 δ1↑ | KITTI δ1↑ | ETH3D δ1↑ |
|--------|----------|----------|----------|
| DINOv3 + DAv2 | **98.0** | **96.7** | **97.5** |
| DAv2 (ViT-g) | 97.9 | 94.7 | 86.5 |
| Marigold | 96.4 | 91.6 | 96.0 |

This validates that DINOv3 inherits DINOv2's strong **sim-to-real** capabilities —
trained on synthetic data, it generalizes perfectly to real-world images.
The DPT decoder uses features from 4 intermediate layers [10, 20, 30, 40].

## 4. Object Detection (Section 6.3.1)

DINOv3 with Plain-DETR achieves **66.1 mAP** on COCO — the **first competitive detector**
to use a completely frozen backbone. All prior SOTA models finetune their backbones.

**Key Results (Table 10)**:

| Model | Detector | Backbone FT? | Trainable | COCO mAP | COCO-O mAP |
|-------|----------|-------------|-----------|----------|------------|
| DINOv3 | Plain-DETR | No (frozen) | 100M | 66.1 | 66.4 |
| PEspatial | DETA | Yes | 2B | 66.0 | 64.0 |
| EVA-02 | Co-DETR | Yes | 300M | 65.9 | 63.7 |

With only 100M trainable decoder parameters, DINOv3 surpasses models that finetune
their entire backbone (300M-2B params). On the OOD benchmark COCO-O, the gap widens further.

## 5. Video Segmentation Tracking (Section 6.1.5)

DINOv3 achieves **83.3 J&F** on DAVIS at high resolution — despite being trained
only on images, never on video. This uses a simple non-parametric label propagation
algorithm based on patch feature cosine similarity across frames.

**Key Results (Table 5)**:

| Model | DAVIS-L | YouTube-VOS-L | MOSE-L |
|-------|---------|---------------|--------|
| DINOv3 | **83.3** | **80.7** | **55.6** |
| DINOv2 | 76.6 | 74.6 | 48.5 |
| AM-RADIO | 81.4 | 79.2 | 54.3 |

Key insight: performance **improves with resolution** for DINOv3, while it degrades for
PEspatial and stays flat for SigLIP 2. This confirms that DINOv3's high-resolution
features (enabled by RoPE + Gram anchoring) are genuinely useful.

## 6. 3D Correspondence Estimation (Section 6.1.3)

DINOv3 features exhibit strong **multi-view consistency** — patches of the same
3D keypoint in different views have high feature similarity.

**Key Results (Table 4)**:

| Model | NAVI (Geometric) | SPair (Semantic) |
|-------|-----------------|------------------|
| DINOv3 | **64.4** | **58.7** |
| DINOv2 | 60.1 | 56.1 |
| AM-RADIO | 59.4 | 56.8 |
| SigLIP 2 | 49.4 | 42.6 |

This makes DINOv3 an excellent backbone for 3D applications like VGGT (Table 13),
where swapping DINOv2→DINOv3 improved results on all benchmarks.

## 7. Unsupervised Object Discovery (Section 6.1.4)

Using TokenCut (a graph-based algorithm that segments objects using patch similarity),
DINOv3 discovers objects **without any annotations**.

**Key Results (Figure 14)**:

| Model | VOC07 CorLoc | VOC12 CorLoc | COCO CorLoc |
|-------|-------------|-------------|-------------|
| DINOv3 | **66.1** | **69.5** | **55.1** |
| DINO (original) | 61.1 | 66.0 | 48.7 |
| DINOv2 | 55.6 | 60.4 | 45.4 |
| Web-DINO 7B | 26.1 | 29.7 | 20.9 |

Interestingly, DINOv2 actually regressed from the original DINO on this task — its
dense feature artifacts hurt object discovery. DINOv3 fixes this with Gram anchoring.

## Summary: DINOv3 vs. The Competition

| Task Type | Best SSL | Best WSL | DINOv3 | Winner |
|-----------|---------|---------|--------|--------|
| Segmentation (ADE20k linear) | DINOv2: 49.5 | AM-RADIO: 53.0 | **55.9** | DINOv3 |
| Depth (NYUv2 RMSE↓) | DINOv2: 0.372 | PEspatial: 0.362 | **0.309** | DINOv3 |
| Tracking (DAVIS J&F) | DINOv2: 76.6 | AM-RADIO: 81.4 | **83.3** | DINOv3 |
| Keypoints (NAVI recall) | DINOv2: 60.1 | AM-RADIO: 59.4 | **64.4** | DINOv3 |
| Object Discovery (VOC07) | DINO: 61.1 | AM-RADIO: 55.0 | **66.1** | DINOv3 |
| Classification (IN-1k) | DINOv2: 87.3 | PE: 89.3 | **88.4** | PE |
| Retrieval (Oxford-H) | DINOv2: 58.2 | AM-RADIO: 50.7 | **60.7** | DINOv3 |

**Bottom line**: DINOv3 dominates on dense tasks and is competitive on global tasks.
A single frozen backbone serves as a universal visual encoder across all these tasks.